# 01. LLM 출력을 `eval()`로 실행할 때의 취약점 (OWASP LLM10 / ASI05)

| 항목 | 내용 |
|---|---|
| OWASP | **LLM10:2026 Improper Output Handling** (LLM 출력을 검증 없이 다음 단계로 넘김) / **ASI05 Unexpected Code Execution** |
| 원본 위치 | `00langchainbasic/001-2_OutPutParser종류.ipynb` — `safe_execute_pandas()` |
| 위험도 | 높음 (임의 코드 실행, RCE: Remote Code Execution) |
| LLM 호출 | 없음 — LLM 응답을 문자열로 모의하므로 API 키·비용이 필요 없습니다 |

## 핵심 개념
- LLM의 출력은 **사용자 입력과 같은 "신뢰할 수 없는 입력"** 입니다. (프롬프트 인젝션으로 얼마든지 조작될 수 있음)
- 원본 코드는 "허용 패턴 목록"으로 걸러낸 뒤 `eval()`을 호출해서 안전해 보이지만, 패턴 검사가 `re.match`(**문자열 앞부분만** 검사)라서 뒤에 붙은 악성 코드를 막지 못합니다.

## 이 노트북의 흐름
1. Before: 원본 코드 그대로
2. 공격 재현 (무해한 페이로드 사용)
3. After: `eval()` 없이 AST(Abstract Syntax Tree, 추상 구문 트리)로 허용 문법만 해석
4. 같은 입력으로 전/후 비교 + 자동 검증(`assert`)

In [ ]:
import ast
import re

import pandas as pd

# Titanic 데이터셋 로드 (원본과 동일)
DATA_PATH = "../../data/titanic.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape, list(df.columns)[:5])

## 1. Before — 취약한 코드 (원본 그대로)

In [ ]:
def safe_execute_pandas_before(df, command: str):
    """[취약] 허용된 pandas 명령어만 실행한다고 가정하지만 실제로는 앞부분만 검사"""
    safe_commands = {
        r"df\['[^']*'\]": lambda: eval(command, {"df": df}, {}),
        r'df\["[^"]*"\]': lambda: eval(command, {"df": df}, {}),
        r"df\.iloc\[\d+\]": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.mean\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.shape": lambda: eval(command, {"df": df}, {}),
    }

    command = command.strip()
    for pattern, executor in safe_commands.items():
        if re.match(pattern, command):   # <-- 문제: 앞부분만 일치하면 통과
            try:
                return executor()        # <-- 문제: 문자열 "전체"를 eval 로 실행
            except Exception as e:
                return f"실행 오류: {e}"
    return "허용되지 않은 명령어입니다."

## 2. 공격 재현

LLM이 아래 문자열을 만들어 냈다고 가정합니다. (프롬프트 인젝션, 오염된 문서 등으로 충분히 가능)
실제 공격이라면 `os.system(...)`, 파일 삭제, 키 유출 등이 들어가지만, 여기서는 **화면에 출력하거나 현재 경로를 읽는 무해한 코드**만 사용합니다.

In [ ]:
# 정상 명령 (LLM이 의도대로 만든 응답)
normal_commands = [
    "df['Name'].head()",
    "df.iloc[0]",
    "df['Fare'].mean()",
    "df.head()",
    "df.shape",
]

# 공격 명령 — 모두 "허용 패턴으로 시작"하므로 원본 검사를 통과한다
attack_commands = [
    # 1) 허용 패턴(df.shape) 뒤에 임의 코드를 붙임 → 현재 작업 경로 유출
    "df.shape and __import__('os').getcwd()",
    # 2) 임의 코드 실행 사실을 화면에 출력 (os.system 등으로 바꾸면 실제 피해)
    "df.shape and print('🚨 임의 코드가 실행되었습니다')",
    # 3) 객체 내부 구조 탐색 (샌드박스 탈출 시도의 첫 단계)
    "df['Name'].__class__.__mro__",
]

print("[정상 명령]")
for cmd in normal_commands[:3]:
    print(f"  {cmd!r} -> {type(safe_execute_pandas_before(df, cmd)).__name__}")

print("\n[공격 명령]  ※ 모두 '막혀야' 정상인데 실행되는지 확인")
for cmd in attack_commands:
    result = safe_execute_pandas_before(df, cmd)
    print(f"  {cmd!r}\n    -> 결과: {result!r}")

> **결과 해석**: 3개 모두 차단되지 않고 실행됩니다.
> 원인은 두 가지입니다. ① `re.match` 는 문자열의 **시작만** 확인합니다. ② `eval()` 은 **문자열 전체**를 파이썬 코드로 실행합니다.
> "패턴으로 먼저 거른다"는 방식은 **허용 목록(allowlist)처럼 보이지만 실제로는 구멍이 있는 필터**입니다.

## 3. After — `eval()`을 없애고 허용 문법만 직접 해석

수정 원칙
1. **`eval()`/`exec()` 사용 금지** — 문자열을 코드로 실행하지 않습니다.
2. 문자열을 `ast.parse` 로 **구문 트리**로 만든 뒤, 허용한 노드(`df`, `df['컬럼']`, `df.iloc[정수]`, `.mean()`, `.head(n)`, `df.shape`)만 **직접 계산**합니다.
3. 컬럼 이름은 **실제 `df.columns` 안에 있을 때만** 허용하고, 길이·인자 범위도 제한합니다.
4. 허용하지 않은 것은 모두 거부합니다. (기본값 = 거부)

In [ ]:
MAX_COMMAND_LEN = 200          # 명령 길이 상한
MAX_HEAD_ROWS = 100            # head(n) 의 n 상한
ALLOWED_METHODS = {"mean", "head"}   # 허용할 메서드 이름


class UnsafeCommandError(ValueError):
    """허용되지 않은 명령일 때 발생"""


def _eval_node(node, df):
    """허용한 문법만 계산하고, 그 외는 UnsafeCommandError 를 발생시킨다 (eval 미사용)."""
    # df
    if isinstance(node, ast.Name) and node.id == "df":
        return df

    # df.shape
    if (isinstance(node, ast.Attribute) and isinstance(node.value, ast.Name)
            and node.value.id == "df" and node.attr == "shape"):
        return df.shape

    # df['컬럼']  /  df.iloc[정수]
    if isinstance(node, ast.Subscript):
        idx = node.slice
        if not isinstance(idx, ast.Constant):
            raise UnsafeCommandError("인덱스는 상수만 허용")
        base = node.value
        if isinstance(base, ast.Name) and base.id == "df":
            if isinstance(idx.value, str) and idx.value in df.columns:
                return df[idx.value]
            raise UnsafeCommandError(f"존재하지 않는 컬럼: {idx.value!r}")
        if (isinstance(base, ast.Attribute) and base.attr == "iloc"
                and isinstance(base.value, ast.Name) and base.value.id == "df"):
            if isinstance(idx.value, int) and not isinstance(idx.value, bool) and 0 <= idx.value < len(df):
                return df.iloc[idx.value]
            raise UnsafeCommandError("iloc 인덱스는 범위 안의 정수만 허용")

    # <허용된 식>.mean()  /  <허용된 식>.head(n)
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
        if node.func.attr not in ALLOWED_METHODS or node.keywords:
            raise UnsafeCommandError(f"허용되지 않은 메서드/인자: {node.func.attr}")
        target = _eval_node(node.func.value, df)     # 대상도 재귀적으로 검증
        if node.func.attr == "mean" and not node.args:
            return target.mean()
        if node.func.attr == "head" and len(node.args) <= 1:
            n = 5
            if node.args:
                a = node.args[0]
                if not (isinstance(a, ast.Constant) and isinstance(a.value, int)
                        and not isinstance(a.value, bool) and 1 <= a.value <= MAX_HEAD_ROWS):
                    raise UnsafeCommandError(f"head 인자는 1~{MAX_HEAD_ROWS} 정수만 허용")
                n = a.value
            return target.head(n)

    raise UnsafeCommandError(f"허용되지 않은 문법: {type(node).__name__}")


def safe_execute_pandas_after(df, command: str):
    """[개선] AST 로 허용 문법만 해석한다. 실패하면 이유를 문자열로 돌려준다."""
    command = command.strip()
    if len(command) > MAX_COMMAND_LEN:
        return "허용되지 않은 명령어입니다. (너무 김)"
    try:
        tree = ast.parse(command, mode="eval")   # 코드를 실행하지 않고 "구조"만 분석
        return _eval_node(tree.body, df)
    except (SyntaxError, UnsafeCommandError) as e:
        return f"허용되지 않은 명령어입니다. ({e})"
    except Exception as e:                       # 계산 중 일반 오류
        return f"실행 오류: {type(e).__name__}"

## 4. 같은 입력으로 전/후 비교

In [ ]:
def is_blocked(result) -> bool:
    """결과가 거부 메시지인지 판단"""
    return isinstance(result, str) and result.startswith("허용되지 않은 명령어")


rows = []
for kind, cmds in (("정상", normal_commands), ("공격", attack_commands)):
    for cmd in cmds:
        before = safe_execute_pandas_before(df, cmd)
        after = safe_execute_pandas_after(df, cmd)
        rows.append({
            "구분": kind,
            "명령": cmd,
            "Before": "차단" if is_blocked(before) else "실행됨",
            "After": "차단" if is_blocked(after) else "실행됨",
        })

compare = pd.DataFrame(rows)
compare

In [ ]:
# 자동 검증: 정상은 계속 동작하고, 공격은 After 에서 모두 차단되어야 한다
for cmd in normal_commands:
    assert not is_blocked(safe_execute_pandas_after(df, cmd)), f"정상 명령이 막힘: {cmd}"
for cmd in attack_commands:
    assert is_blocked(safe_execute_pandas_after(df, cmd)), f"공격이 통과함: {cmd}"
# Before 는 공격이 실제로 통과했음을 확인 (취약점 재현 확인)
assert all(not is_blocked(safe_execute_pandas_before(df, c)) for c in attack_commands)

print("검증 통과: 정상 명령은 동작, 공격 명령은 After 에서 모두 차단")
print(safe_execute_pandas_after(df, "df.shape and __import__('os').getcwd()"))

## 5. 바뀐 점 요약

| 구분 | Before | After |
|---|---|---|
| 검사 방식 | `re.match` (앞부분만) | `ast.parse` 후 노드별 검사 (전체 구조) |
| 실행 방식 | `eval(문자열)` | 허용 노드를 직접 계산 (eval 없음) |
| 컬럼 검증 | 없음 | 실제 `df.columns` 에 있는 이름만 |
| 인자 제한 | 없음 | `head(n)` 1~100, 길이 200자 |
| 기본 정책 | 패턴에 맞으면 실행 | 허용한 것 외에는 모두 거부 |

## 6. 남는 위험과 추가 대책
- 허용한 동작 자체의 부작용: 컬럼이 아주 크면 응답이 커질 수 있으므로 출력 크기도 제한하세요.
- 더 복잡한 질의(필터, groupby 등)가 필요하면 `eval` 을 늘리지 말고 **미리 만든 함수(도구) 목록**에서 고르게 하세요. (LLM은 "함수 이름 + 인자"만 반환)
- 임의 코드 실행이 꼭 필요하면 컨테이너 등 **격리된 샌드박스**에서 최소 권한으로 실행해야 합니다.
- 사용자 승인(Human-in-the-Loop)과 로그 기록을 함께 두면 더 안전합니다.